In [1]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np

print("All imports done!")

d:\ask-my-docs\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports done!


In [11]:

client_db = chromadb.PersistentClient(path="../data/chromadb_rag")
collection = client_db.get_or_create_collection(name="my_pdf_docs")


all_data = collection.get()
chunks = all_data['documents']

print(f"Total chunks loaded: {len(chunks)}")

Total chunks loaded: 7


In [12]:
tokenized_chunks = [chunk.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index ready!")

BM25 index ready!


In [13]:
query = "morphological operations binary images shapes"

# BM25 Search — keyword based
tokenized_query = query.lower().split()
bm25_scores = bm25.get_scores(tokenized_query)
top_bm25_indices = np.argsort(bm25_scores)[::-1][:5]

# Vector Search — meaning based
vector_results = collection.query(
    query_texts=[query],
    n_results=5
)
top_vector_chunks = vector_results['documents'][0]

# Combine karo dono results
bm25_chunks = [chunks[i] for i in top_bm25_indices]

# Union of both results
combined = list(dict.fromkeys(bm25_chunks + top_vector_chunks))

print(f"BM25 results: {len(bm25_chunks)}")
print(f"Vector results: {len(top_vector_chunks)}")
print(f"Combined unique results: {len(combined)}")

BM25 results: 5
Vector results: 5
Combined unique results: 6


In [14]:
print("=== BM25 Top Result ===")
print(bm25_chunks[0][:300])

print("\n=== Vector Search Top Result ===")
print(top_vector_chunks[0][:300])

print("\n=== Combined Top Result ===")
print(combined[0][:300])

=== BM25 Top Result ===
4. Morphological Operations Morphological operations are used to process images based on shapes. They are mainly applied on binary images. Basic Operations: (a) Erosion ● Removes pixels from object boundaries ● Shrinks objects (b) Dilation ● Adds pixels to boundaries ● Expands objects (c) Opening ● 

=== Vector Search Top Result ===
4. Morphological Operations Morphological operations are used to process images based on shapes. They are mainly applied on binary images. Basic Operations: (a) Erosion ● Removes pixels from object boundaries ● Shrinks objects (b) Dilation ● Adds pixels to boundaries ● Expands objects (c) Opening ● 

=== Combined Top Result ===
4. Morphological Operations Morphological operations are used to process images based on shapes. They are mainly applied on binary images. Basic Operations: (a) Erosion ● Removes pixels from object boundaries ● Shrinks objects (b) Dilation ● Adds pixels to boundaries ● Expands objects (c) Opening ● 
